# 🔍 HolmHz — Colab Training Template

**Hệ thống Phát hiện Ảnh Tổng hợp (Synthetic Image Detection)**

Notebook này dùng để train/evaluate model HolmHz trên Google Colab với GPU miễn phí.

## Workflow:
1. Mount Google Drive (lưu data + checkpoints)
2. Clone repo & cài dependencies
3. Import `holmhz` package
4. Train / Evaluate / Visualize

## 1️⃣ Mount Google Drive

Google Drive sẽ là nơi lưu:
- **Dataset** (`data/`) — không cần re-upload mỗi lần
- **Checkpoints** (`outputs/`) — model weights không mất khi runtime disconnect
- **Source code** (`HolmHz/`) — clone từ GitHub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục làm việc trên Drive
import os
DRIVE_ROOT = '/content/drive/MyDrive/HolmHz'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'✅ Drive mounted. Working dir: {DRIVE_ROOT}')

## 2️⃣ Kiểm tra GPU

In [ ]:
# Kiểm tra GPU được cấp (Colab: Runtime > Change runtime type > GPU)
!nvidia-smi

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 3️⃣ Clone Repo & Cài Dependencies

In [ ]:
import os

REPO_DIR = '/content/HolmHz'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/EurusDevSec/HolmHz.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'✅ Working directory: {os.getcwd()}')

In [ ]:
# Cài holmhz package (editable mode) + dependencies
!pip install -e . --quiet

# Verify import
import holmhz
print(f'✅ holmhz v{holmhz.__version__} installed successfully')

## 4️⃣ Setup W&B + Environment

In [ ]:
import wandb

# Cách 1: Login bằng API key (paste khi được hỏi)
wandb.login()

# Cách 2: Dùng Colab secrets (an toàn hơn)
# from google.colab import userdata
# wandb.login(key=userdata.get('WANDB_API_KEY'))

## 5️⃣ Symlink Data từ Google Drive

Thay vì copy data vào Colab (mất thời gian), ta tạo symlink từ Drive → repo.

**Lần đầu**: Upload data lên Google Drive theo cấu trúc:
```
Google Drive/HolmHz/
├── data/
│   ├── processed/train/
│   ├── processed/val/
│   └── manifests/
└── outputs/
    └── checkpoints/
```

In [ ]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/HolmHz'
REPO_DIR = '/content/HolmHz'

# Symlink data từ Drive vào repo
links = {
    f'{DRIVE_ROOT}/data': f'{REPO_DIR}/data',
    f'{DRIVE_ROOT}/outputs': f'{REPO_DIR}/outputs',
    f'{DRIVE_ROOT}/weights': f'{REPO_DIR}/weights',
}

for src, dst in links.items():
    os.makedirs(src, exist_ok=True)
    if os.path.exists(dst):
        os.remove(dst) if os.path.islink(dst) else None
    if not os.path.exists(dst):
        os.symlink(src, dst)
        print(f'✅ {dst} -> {src}')

print('\n✅ Data linked from Google Drive!')

## 6️⃣ Quick Sanity Check

In [ ]:
# Kiểm tra mọi thứ sẵn sàng
import torch
import timm
import holmhz
from pathlib import Path

checks = {
    'PyTorch': torch.__version__,
    'CUDA': torch.cuda.is_available(),
    'GPU': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A',
    'timm': timm.__version__,
    'holmhz': holmhz.__version__,
    'data dir': Path('data').exists(),
    'outputs dir': Path('outputs').exists(),
}

for k, v in checks.items():
    status = '✅' if v and v != 'N/A' else '❌'
    print(f'{status} {k}: {v}')

print('\n🚀 Ready to train!' if all(checks.values()) else '\n⚠️ Some checks failed')

## 7️⃣ Training (sử dụng khi đã có data + model code)

```python
# Option 1: Chạy script trực tiếp
!python scripts/train.py --config configs/train.yaml

# Option 2: Import và dùng trong notebook
from holmhz.training.trainer import Trainer
from omegaconf import OmegaConf

cfg = OmegaConf.load('configs/train.yaml')
trainer = Trainer(cfg)
trainer.train()
```